In [1]:
from google.colab import userdata
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%cd "/content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Wahib B/cleaned_json_full"

/content/drive/.shortcut-targets-by-id/1Yk-plf3-NMUk7VUGN3mnz81xvzAIAyKX/PIP_2025-2026_Groupe-1_Concours/Wahib B/cleaned_json_full


In [ ]:
# 1 — Installation des dépendances
!pip install -q \
    transformers \
    sentence-transformers \
    faiss-cpu \
    pymupdf \
    beautifulsoup4 \
    accelerate


In [ ]:
# =========================
# Reranker — CrossEncoder
# =========================

from sentence_transformers import CrossEncoder
import torch

print("🔁 Initialisation du reranker (CrossEncoder)...")

def load_reranker(model_name: str = "cross-encoder/ms-marco-MiniLM-L-6-v2"):
    """
    Charge un reranker CrossEncoder avec gestion automatique GPU / CPU.
    Retourne None si le chargement échoue (fallback propre).
    """

    try:
        device = "cuda" if torch.cuda.is_available() else "cpu"

        reranker = CrossEncoder(
            model_name,
            device=device,
            max_length=512    # 🔒 limite mémoire, suffisant pour QA
        )

        print(f"✅ Reranker chargé sur {device.upper()} : {model_name}")
        return reranker

    except Exception as e:
        print("⚠️ Impossible de charger le reranker.")
        print(f"🛠️ Détail : {e}")
        print("➡️ Le système continuera sans reranking (FAISS seul).")
        return None


# Chargement effectif
reranker = load_reranker()

🔁 Initialisation du reranker (CrossEncoder)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


✅ Reranker chargé sur CUDA : cross-encoder/ms-marco-MiniLM-L-6-v2


In [ ]:
# =========================
# 2 — Imports & paramètres globaux (NETTOYÉ)
# =========================

# ----- Standard library -----
import os
import sys
import pickle
from typing import List, Dict

# ----- Calcul scientifique -----
import numpy as np

# ----- GPU / Deep Learning -----
import torch

# ----- Traitement documents -----
import fitz  # PyMuPDF (PDF)
from bs4 import BeautifulSoup  # HTML

# ----- Recherche vectorielle -----
import faiss

# ----- Embeddings & reranking -----
from sentence_transformers import SentenceTransformer, CrossEncoder

# ----- LLM / Hugging Face -----
# Note : On utilise Unsloth pour le modèle, mais ces imports restent utiles pour les outils
from transformers import (
    AutoTokenizer,
    pipeline
)

# =========================
# Paramètres globaux
# =========================

# Mode debug (logs détaillés)
DEBUG = False

# Seuils & constantes de sécurité
MIN_TEXT_LENGTH = 50         # Ignore les textes trop courts
MIN_QUERY_LENGTH = 3         # Ignore les questions trop vagues

# Mémoire & performances
DEFAULT_BATCH_GPU = 8
DEFAULT_BATCH_CPU = 4

# Répertoire de travail
PROJECT_ROOT = "/content"
FAISS_INDEX_DIR = os.path.join(PROJECT_ROOT, "faiss_index_cnrs")

# Création du dossier FAISS si nécessaire
os.makedirs(FAISS_INDEX_DIR, exist_ok=True)

print("✅ Imports et paramètres globaux chargés (Sans casser Unsloth).")

✅ Imports et paramètres globaux chargés (Sans casser Unsloth).


In [ ]:
# =========================
# PARAMÈTRES (OPTIMISÉ POUR QWEN & UNSLOTH)
# =========================

DATA_DIR = "/content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Wahib B/cleaned_json_full"

# On met à jour le nom pour être cohérent, même si le chargement se fait via Unsloth
MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
load_in_4bit = True

EMBEDDING_MODEL = "sentence-transformers/all-mpnet-base-v2"

# Chunking (OK pour JSON structuré)
CHUNK_SIZE = 400
OVERLAP = 100

# Retrieval
TOP_K = 8
SIMILARITY_THRESHOLD = 0.30

# 🔒 Sécurité mémoire LLM
# 🚀 J'ai augmenté ces limites pour que le jury lise plus de contexte
MAX_CHUNKS_FOR_LLM = 5        # On donne plus de morceaux de texte au modèle
MAX_CONTEXT_CHARS = 3500      # 700 était trop court, 3500 permet de lire une page complète

# Génération
MAX_NEW_TOKENS = 300          # Permet des réponses un peu plus complètes
TEMPERATURE = 0.2
REPETITION_PENALTY = 1.1

In [ ]:
# =========================
# 3 — Détection GPU automatique
# =========================

import torch

def can_use_gpu(min_free_gb: float = 4.0, verbose: bool = True) -> bool:
    """
    Détermine si le GPU peut être utilisé en toute sécurité.

    Paramètres :
    - min_free_gb : mémoire GPU libre minimale requise (en Go)
    - verbose : affiche des informations détaillées

    Retour :
    - True si le GPU est utilisable
    - False sinon (fallback CPU)
    """

    # 1️⃣ CUDA disponible ?
    if not torch.cuda.is_available():
        if verbose:
            print("⚠️ CUDA non disponible → mode CPU")
        return False

    try:
        # 2️⃣ Mémoire GPU disponible
        free_bytes, total_bytes = torch.cuda.mem_get_info()
        free_gb = free_bytes / (1024 ** 3)
        total_gb = total_bytes / (1024 ** 3)

        # 3️⃣ Informations GPU
        device_name = torch.cuda.get_device_name(0)

        if verbose:
            print(f"🖥️ GPU détecté : {device_name}")
            print(f"💾 Mémoire GPU libre : {free_gb:.2f} Go / {total_gb:.2f} Go")

        # 4️⃣ Seuil minimal requis
        if free_gb < min_free_gb:
            if verbose:
                print(
                    f"⚠️ Mémoire GPU insuffisante "
                    f"(min requis : {min_free_gb} Go) → mode CPU"
                )
            return False

        return True

    except Exception as e:
        if verbose:
            print(f"❌ Erreur lors de la détection GPU : {e}")
            print("➡️ Fallback CPU")
        return False


# =========================
# Sélection automatique du mode
# =========================

USE_GPU = can_use_gpu(min_free_gb=4.0)

print(
    f"\n🔍 Mode sélectionné : "
    f"{'🟢 GPU' if USE_GPU else '🟡 CPU'}\n"
)


🖥️ GPU détecté : NVIDIA A100-SXM4-40GB
💾 Mémoire GPU libre : 39.04 Go / 39.56 Go

🔍 Mode sélectionné : 🟢 GPU



In [ ]:
# =========================
# 4 — Chargement et préparation des documents
#     (DOSSIER JSON STRUCTURÉ – VERSION FINALE)
# =========================

import os
import json
import re


# ---------- Nettoyage ----------
def clean_text(text: str) -> str:
    text = text.replace("\x00", "")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n", text)
    return text.strip()


# ---------- Normalisation ----------
def normalize_text(text: str) -> str:
    text = text.replace("•", "- ")
    text = text.replace("–", "- ")
    text = text.replace("—", "- ")
    text = text.replace("* ", "- ")
    return text


# ---------- Construction de chunks logiques ----------
def build_chunks_from_poste(base_meta: dict, poste: dict):
    """
    Crée des chunks séparés pour mission, activités, compétences, contexte.
    """
    chunks = []

    def add_chunk(label, content):
        if content and len(content.strip()) > 50:
            chunks.append({
                "text": f"{label} : {normalize_text(clean_text(content))}",
                **base_meta
            })

    add_chunk("Mission", poste.get("mission"))
    add_chunk("Activités", poste.get("activites"))
    add_chunk("Compétences", poste.get("competences"))
    add_chunk("Contexte", poste.get("contexte"))

    return chunks


# =========================
# CONSTRUCTION DU CORPUS
# =========================

documents = []

print("📂 Chargement des documents JSON structurés...")

for filename in os.listdir(DATA_DIR):
    if not filename.lower().endswith(".json"):
        continue

    path = os.path.join(DATA_DIR, filename)

    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        # Métadonnées communes au concours
        base_meta_common = {
            "source": filename,
            "bap": data.get("bap"),
            "grade": data.get("grade"),
            "concours_label": data.get("concours_label"),
            "concours_num": data.get("concours_num"),
            "nb_postes": data.get("nb_postes"),
        }

        # Boucle sur les postes
        for poste in data.get("postes", []):
            base_meta = {
                **base_meta_common,
                "poste_num": poste.get("poste_num"),
                "affectation": poste.get("affectation"),
                "groupe_fonction": poste.get("groupe_fonction"),
            }

            chunks = build_chunks_from_poste(base_meta, poste)
            documents.extend(chunks)

    except Exception as e:
        print(f"⚠️ Erreur JSON {filename} : {e}")

print(f"✅ Documents indexés : {len(documents)} chunks")


📂 Chargement des documents JSON structurés...
✅ Documents indexés : 592 chunks


In [ ]:
# =========================
# 5 — Embeddings & FAISS (CORRIGÉ)
# =========================

from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import os
import torch

print("🔎 Initialisation du modèle d'embeddings...")

# 1️⃣ Chargement du modèle d'embeddings
# CORRECTION : Détection automatique du GPU
device = "cuda" if torch.cuda.is_available() else "cpu"

embedder = SentenceTransformer(
    "sentence-transformers/all-mpnet-base-v2",
    device=device
)

# 2️⃣ Préparation des textes
# Assure-toi que la variable 'documents' existe (créée dans une cellule précédente)
texts = [d["text"] for d in documents if d.get("text")]

if not texts:
    raise ValueError("❌ Aucun texte valide pour la vectorisation.")

# 3️⃣ Encodage par batch
print(f"📐 Calcul des embeddings pour {len(texts)} chunks...")

embeddings = embedder.encode(
    texts,
    # On ajuste le batch selon si on est sur GPU ou CPU
    batch_size=16 if device == "cuda" else 4,
    normalize_embeddings=True,
    show_progress_bar=True
)

# 4️⃣ Conversion FAISS (float32 obligatoire)
embeddings = np.asarray(embeddings, dtype="float32")

# 5️⃣ Création de l'index FAISS (cosine similarity via IP)
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)

# Sécurité : vérifier cohérence
assert index.is_trained, "Index FAISS non entraîné"

# 6️⃣ Ajout des vecteurs
index.add(embeddings)

print(f"✅ Index FAISS créé avec {index.ntotal} vecteurs sur {device.upper()}.")

🔎 Initialisation du modèle d'embeddings...
📐 Calcul des embeddings pour 592 chunks...


Batches:   0%|          | 0/37 [00:00<?, ?it/s]

✅ Index FAISS créé avec 592 vecteurs sur CUDA.


In [ ]:
def handle_small_talk(question: str):
    q = question.lower().strip()

    greetings = ["bonjour", "bonsoir", "salut", "hello", "hi"]
    thanks = ["merci", "merci beaucoup", "thanks"]

    if q in greetings:
        return (
            "Bonjour 👋\n"
            "Je suis l’agent d’information sur les concours ingénieurs du CNRS.\n"
            "Vous pouvez me poser des questions sur les concours, postes, missions ou compétences."
        )

    if q in thanks:
        return "Avec plaisir 😊 N’hésitez pas si vous avez d’autres questions sur les concours CNRS."

    return None


In [ ]:
from collections import Counter

def filter_same_source(chunks):
    """
    Garde uniquement les chunks provenant majoritairement
    du même fichier (même concours).
    """
    if not chunks:
        return []

    sources = [c["source"] for c in chunks]
    main_source = Counter(sources).most_common(1)[0][0]

    return [c for c in chunks if c["source"] == main_source]


In [ ]:
def is_orientation_question(question: str) -> bool:
    keywords = [
        "correspond", "profil", "je suis", "quel concours",
        "orienter", "adapté", "accessible avec"
    ]
    q = question.lower()
    return any(k in q for k in keywords)


In [ ]:
# =========================
# 6 — Retrieval avec reranking
# =========================

def retrieve(question: str):
    """
    Recherche les passages les plus pertinents pour une question donnée.
    Étapes :
    1. Retrieval large via FAISS (rappel)
    2. Filtrage par seuil de similarité
    3. Reranking précis (cross-encoder)
    4. Sélection finale TOP_K
    """

    # 🔒 Sécurité minimale
    if not question or len(question.strip()) < 3:
        return []

    # 1️⃣ Embedding de la question
    try:
        q_emb = embedder.encode(
            [question],
            normalize_embeddings=True
        )
    except Exception:
        return []

    # 2️⃣ Retrieval large (on sur-échantillonne pour le reranker)
    try:
        scores, indices = index.search(q_emb, TOP_K * 3)
    except Exception:
        return []

    # 3️⃣ Filtrage par similarité FAISS
    candidates = []
    for score, idx in zip(scores[0], indices[0]):

        # Index invalide
        if idx < 0 or idx >= len(documents):
            continue

        # Seuil anti-hallucination
        if score < SIMILARITY_THRESHOLD:
            continue

        doc = documents[idx].copy()
        doc["faiss_score"] = float(score)
        candidates.append(doc)

    # Aucun contexte fiable
    if not candidates:
        return []

    # 4️⃣ Reranking précis (si disponible)
    if reranker is not None:
        try:
            pairs = [(question, c["text"]) for c in candidates]
            rerank_scores = reranker.predict(pairs)

            for c, r_score in zip(candidates, rerank_scores):
                c["rerank_score"] = float(r_score)

            # Combinaison FAISS + reranker
            reranked = sorted(
                candidates,
                key=lambda c: (c["rerank_score"], c["faiss_score"]),
                reverse=True
            )
        except Exception:
            # Fallback : FAISS seul
            reranked = sorted(
                candidates,
                key=lambda c: c["faiss_score"],
                reverse=True
            )
    else:
        reranked = sorted(
            candidates,
            key=lambda c: c["faiss_score"],
            reverse=True
        )

    # 5️⃣ Sélection finale (TOP_K)
    return reranked[:TOP_K]


In [ ]:
# =========================
# 7 — Chargement du modèle LLM (Qwen/Qwen2.5-14B-Instruct)
# =========================

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch
import os

print("🚀 Chargement du modèle LLM...")

# 1️⃣ Chargement du tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True
)

# 2️⃣ Chargement du modèle avec gestion GPU / CPU
if USE_GPU and torch.cuda.is_available():

    print("🟢 GPU détecté — Chargement en mode GPU optimisé")

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map="auto",                  # Placement automatique GPU / CPU
        dtype=torch.float16,                # Réduction mémoire (corrected from torch_dtype)
        load_in_4bit=True,                  # Explicitly load in 4-bit for bnb model
        low_cpu_mem_usage=True,
        max_memory={
            0: "14GiB",                     # GPU T4 ≈ 16 Go
            "cpu": "10GiB"
        }
    )

else:
    print("🟡 GPU non disponible — Chargement en mode CPU (secours)")

    os.environ["CUDA_VISIBLE_DEVICES"] = ""
    torch.set_num_threads(4)

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map={"": "cpu"},
        dtype=torch.float32,               # Corrected from torch_dtype
        low_cpu_mem_usage=True
        # No load_in_4bit for CPU, it often implies bnb quantization on GPU
    )

# 3️⃣ Mise en mode évaluation (important)
model.eval()

# 4️⃣ Pipeline de génération STRICT (anti-hallucination)
pipe = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=MAX_NEW_TOKENS,

    # 🔑 CLÉ DU SUCCÈS
    do_sample=False,        # Toujours déterministe
    temperature=0.2,        # ⚠️ PAS 0.0
    top_p=1.0,              # On ne tronque plus le raisonnement

    repetition_penalty=1.1, # Évite le blabla

    return_full_text=False,
    pad_token_id=tokenizer.eos_token_id
)

print("✅ Modèle prêt pour l'inférence.")


🚀 Chargement du modèle LLM...
🟢 GPU détecté — Chargement en mode GPU optimisé


PackageNotFoundError: No package metadata was found for bitsandbytes

In [ ]:
# =========================
# 8 — Prompts CNRS (CORRIGÉ & OPTIMISÉ)
# =========================

# ⚠️ On définit la phrase de refus ici pour éviter le crash
REFUS = "Je suis navré, mais je ne trouve pas cette information dans les documents officiels du CNRS fournis."

def build_prompt(question: str, contexts: list) -> str:
    """
    Construit le prompt pour le mode RAG classique (Facts).
    """
    context_text = "\n".join(
        f"- {c['text']}" for c in contexts
    )

    # On structure le prompt pour que Qwen comprenne bien la séparation
    return f"""CONTEXTE OFFICIEL :
{context_text}

INSTRUCTIONS :
Réponds à la question ci-dessous en utilisant UNIQUEMENT le contexte ci-dessus.
Si la réponse n'y est pas, dis exactement : "{REFUS}".
Ne cite pas tes connaissances générales, base-toi sur les textes.

QUESTION :
{question}
"""

def build_orientation_prompt(question, contexts):
    """
    Construit le prompt pour le mode Orientation (Conseil).
    """
    context_text = "\n".join(
        f"- {c['text']}" for c in contexts
    )

    return f"""DOCUMENTS DE RÉFÉRENCE :
{context_text}

MISSION :
Tu es un expert RH du CNRS. Ta mission est d'aider un candidat à s'orienter.
Analyse les documents fournis pour voir si le profil ou la demande du candidat correspond à un concours ou une branche.

RÈGLES :
- Sois nuancé ("il semble que", "ce profil pourrait correspondre à").
- Cite les éléments précis des textes qui te font dire ça.
- Si rien ne correspond, dis-le poliment.

PROFIL / QUESTION DU CANDIDAT :
{question}
"""

def build_search_query(question: str) -> str:
    """
    Transforme une question d'orientation en mots-clés pour la recherche FAISS.
    """
    q = question.lower()
    keywords = []

    # Mapping intelligent des termes
    if "statistique" in q or "data" in q or "données" in q:
        keywords.append("analyse de données big data")
    if "biologique" in q or "bio" in q or "vivant" in q:
        keywords.append("sciences biologiques biologie")
    if "informatique" in q or "dev" in q or "logiciel" in q:
        keywords.append("développement informatique logiciel")
    if "chimie" in q or "matériaux" in q:
        keywords.append("sciences chimiques matériaux")
    if "master" in q or "bac+5" in q:
        keywords.append("ingénieur d'études recherche")
    if "doctorat" in q or "thèse" in q or "phd" in q:
        keywords.append("ingénieur de recherche")

    # Si aucun mot clé trouvé, on garde la question brute
    if not keywords:
        return question

    return " ".join(keywords)

print("✅ Prompts et outils de requête prêts.")

✅ Prompts et outils de requête prêts.


In [ ]:
# =========================
# 9 — Fonction answer() (Version Unsloth)
# =========================
import torch

# 1. Petites fonctions utilitaires (au cas où elles manquent)
def handle_small_talk(question):
    # Détection basique de politesse
    q = question.lower().strip()
    if q in ["bonjour", "hello", "salut", "ça va ?"]:
        return "Bonjour ! Je suis l'assistant jury du CNRS. Comment puis-je vous aider sur les concours ?"
    return None

def is_orientation_question(question):
    # Si la question parle de profil, de CV ou d'orientation
    keywords = ["profil", "correspond", "orienter", "cv", "mon parcours", "suis-je éligible"]
    return any(k in question.lower() for k in keywords)

def filter_same_source(contexts):
    # Garde les contextes uniques pour éviter les doublons
    seen = set()
    unique = []
    for c in contexts:
        # On utilise le début du texte comme empreinte unique
        sig = c["text"][:100]
        if sig not in seen:
            seen.add(sig)
            unique.append(c)
    return unique

# 2. La Fonction Principale
def answer(question: str):
    print(f"📝 Question reçue : {question}")

    # A) Small talk
    st = handle_small_talk(question)
    if st:
        return {"response": st, "sources": [], "has_answer": True}

    # B) Détection du type de question
    orientation = is_orientation_question(question)

    # C) Recherche documentaire (RAG)
    # Si c'est de l'orientation, on cherche des mots clés, sinon la question brute
    search_query = build_search_query(question) if orientation else question

    # On récupère les documents (fonction retrieve définie plus haut)
    contexts = retrieve(search_query)

    # Si aucun document trouvé
    if not contexts:
        return {"response": REFUS, "sources": [], "has_answer": False}

    # D) Nettoyage des documents
    contexts = filter_same_source(contexts)

    # E) Construction du Prompt
    if orientation:
        prompt_content = build_orientation_prompt(question, contexts)
        system_msg = "Tu es un expert RH du CNRS. Aide le candidat à s'orienter."
    else:
        prompt_content = build_prompt(question, contexts)
        system_msg = "Tu es un agent officiel du CNRS. Réponds strictement selon le contexte."

    # F) GÉNÉRATION (C'est ici qu'on utilise Unsloth/Qwen !)
    # On utilise ta nouvelle fonction qwen_chat au lieu de 'pipe'
    try:
        output = qwen_chat(
            system=system_msg,
            user=prompt_content,
            temperature=0.2
        )
    except Exception as e:
        return {"response": f"Erreur lors de la génération : {str(e)}", "sources": [], "has_answer": False}

    # G) Vérification Refus
    if "je ne trouve pas" in output.lower() or REFUS.lower() in output.lower():
        return {"response": REFUS, "sources": [], "has_answer": False}

    # H) Extraction des sources pour l'affichage
    # On essaie de récupérer 'source' et 'page', sinon juste 'source'
    sources_list = []
    for c in contexts:
        src = c.get('source', 'Inconnu')
        page = c.get('page', '')
        if page:
            src += f" (Page {page})"
        sources_list.append(src)

    sources = sorted(list(set(sources_list)))

    return {"response": output, "sources": sources, "has_answer": True}

print("✅ Système de réponse (RAG + Qwen Unsloth) prêt !")

✅ Système de réponse (RAG + Qwen Unsloth) prêt !


In [ ]:
def print_answer(result):
    print("\n📘 RÉPONSE :\n")
    print(result["response"])

    print("\n📂 SOURCES :")
    if not result["sources"]:
        print("Aucune")
    else:
        for s in result["sources"]:
            print(f"- {s}")


In [ ]:
# 🛠️ INSTALLATION D'URGENCE DE UNSLOTH
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
!pip install -U bitsandbytes

In [ ]:
# ==========================================
# 🛑 C'EST ICI LE FINE-TUNING (VERSION CORRIGÉE)
# ==========================================
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template
from trl import SFTTrainer
from transformers import TrainingArguments

# 1. Configuration
max_seq_length = 2048
dtype = None
load_in_4bit = True

print("⏳ Chargement du modèle Qwen 2.5 (Unsloth)...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Configuration LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

# 2. Préparation des données
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5",
    mapping = {"role" : "role", "content" : "content", "user" : "user", "assistant" : "assistant"},
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts, }

# ⚠️ CORRECTION ICI : On pointe directement sur le fichier local
print("📂 Lecture du fichier train.jsonl...")
dataset = load_dataset("json", data_files = "/content/train.jsonl", split = "train")
dataset = dataset.map(formatting_prompts_func, batched = True,)

# 3. LANCEMENT DE L'ENTRAÎNEMENT
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

print("🚀 DÉMARRAGE DU FINE-TUNING MAINTENANT...")
trainer_stats = trainer.train()
print("🎉 FINI ! Ton modèle Qwen a appris le style Jury.")

# Passage en mode utilisation
FastLanguageModel.for_inference(model)

# 4. Fonction de chat pour Qwen (Indispensable pour la suite)
@torch.inference_mode()
def qwen_chat(system: str, user: str, max_new_tokens=450, temperature=0.1):
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize = True,
        add_generation_prompt = True,
        return_tensors = "pt",
    ).to("cuda")

    outputs = model.generate(
        input_ids = inputs,
        max_new_tokens = max_new_tokens,
        use_cache = True,
        temperature = temperature,
        do_sample = True
    )

    response = tokenizer.batch_decode(outputs)
    full_text = response[0]

    # Nettoyage de la réponse
    if "assistant\n" in full_text:
         text_response = full_text.split("assistant\n")[-1].replace("<|im_end|>", "").replace("<|endoftext|>", "").strip()
    else:
         text_response = full_text.split(user)[-1].strip()

    return text_response

⏳ Chargement du modèle Qwen 2.5 (Unsloth)...
==((====))==  Unsloth 2026.1.2: Fast Qwen2 patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


ValueError: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules in 32-bit, you need to set `llm_int8_enable_fp32_cpu_offload=True` and pass a custom `device_map` to `from_pretrained`. Check https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu for more details. 

In [ ]:
def select_contexts_for_llm(contexts, max_chunks=3):
    """
    Sélectionne intelligemment les chunks à envoyer au LLM
    pour éviter les OOM et maximiser la pertinence.
    Priorité : Mission > Compétences > Contexte > Autres
    """
    if not contexts:
        return []

    priority_order = ["Mission", "Compétences", "Contexte", "Activités"]

    selected = []

    for label in priority_order:
        for c in contexts:
            if label.lower() in c["text"].lower() and c not in selected:
                selected.append(c)
                break
        if len(selected) >= max_chunks:
            break

    # Fallback : compléter si pas assez de chunks
    if len(selected) < max_chunks:
        for c in contexts:
            if c not in selected:
                selected.append(c)
            if len(selected) >= max_chunks:
                break

    return selected[:max_chunks]


In [ ]:
def answer(question: str):
    # 1️⃣ Small talk
    st = handle_small_talk(question)
    if st:
        return {
            "response": st,
            "sources": [],
            "has_answer": True
        }

    # 2️⃣ Détection orientation
    orientation = is_orientation_question(question)

    # 3️⃣ Requête de recherche adaptée
    search_query = build_search_query(question) if orientation else question

    # 4️⃣ Retrieval
    contexts = retrieve(search_query)

    if not contexts:
        return {
            "response": REFUS,
            "sources": [],
            "has_answer": False
        }

    # 5️⃣ 🔒 LIMITATION INTELLIGENTE DU CONTEXTE (ANTI-OOM)
    contexts = select_contexts_for_llm(
        contexts,
        max_chunks=MAX_CHUNKS_FOR_LLM
    )

    # 6️⃣ Construction du prompt
    if orientation:
        prompt = build_orientation_prompt(question, contexts)
    else:
        prompt = build_prompt(question, contexts)

    # 7️⃣ Génération
    try:
        output = pipe(prompt)[0]["generated_text"].strip()
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            torch.cuda.empty_cache()
            return {
                "response": "Erreur mémoire GPU. Merci de reformuler la question.",
                "sources": [],
                "has_answer": False
            }
        raise e

    # 8️⃣ Détection refus
    if REFUS.lower() in output.lower():
        return {
            "response": REFUS,
            "sources": [],
            "has_answer": False
        }

    # 9️⃣ Sources propres et dédupliquées
    sources = sorted({
        f"{c['source']} | poste {c.get('poste_num')} | {c.get('affectation')}"
        for c in contexts
    })

    return {
        "response": output,
        "sources": list(sources),
        "has_answer": True
    }


In [ ]:
print("\n" + "=" * 70)
print("🤖 Agent conversationnel CNRS – Concours Ingénieurs")
print("📌 Basé uniquement sur les documents officiels fournis")
print("✋ Tapez 'stop', 'quit' ou 'exit' pour quitter")
print("=" * 70 + "\n")

while True:
    try:
        q = input("❓ Votre question : ").strip()

        if not q:
            print("⚠️ Merci de poser une question.\n")
            continue

        if q.lower() in ["stop", "quit", "exit"]:
            print("\n👋 Fin de la session. Merci !")
            break

        print("\n🔎 Analyse de la question...\n")

        # 🔹 APPEL CORRECT DE answer()
        result = answer(q)

        # 🔹 AFFICHAGE PROPRE
        print_answer(result)

        print("\n" + "-" * 70)

    except KeyboardInterrupt:
        print("\n\n👋 Interruption utilisateur. Fin de la session.")
        break

    except Exception as e:
        print(f"\n❌ Erreur inattendue : {e}")
        print("\n" + "-" * 70)


🤖 Agent conversationnel CNRS – Concours Ingénieurs
📌 Basé uniquement sur les documents officiels fournis
✋ Tapez 'stop', 'quit' ou 'exit' pour quitter


🔎 Analyse de la question...


📘 RÉPONSE :

Basé sur votre profil de titulaire d'un Master en statistiques et data science avec une expérience en analyse de données biologiques, il semble que vous puissiez être bien orienté vers le concours d'Ingénieur de Recherche (IR) en analyse de données -omiques proposé par le Muséum National d'Histoire Naturelle (MNHN) et le CNRS Ecologie & Environnement (CNRS & E&E).

Voici les éléments qui justifient cette orientation :

1. **Adaptation et amélioration d'outils existants** : Votre expérience en analyse de données biologiques peut vous permettre de contribuer à l'adaptation et à l'amélioration des outils existants, comme mentionné dans les activités du poste IR en analyse de données -omiques. Par exemple, vous pouvez concevoir et mettre en œuvre de nouvelles méthodes numériques adaptées à l'hété

In [ ]:
! pip install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 518.9/518.9 kB 10.1 MB/s eta 0:00:00


In [ ]:
# =========================================================
# FINE-TUNING QWEN 2.5 — DATASET INSTRUCTION / INPUT / OUTPUT
# (VERSION STABLE SANS BITSANDBYTES)
# =========================================================

import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments
)
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer

# =========================
# PARAMÈTRES
# =========================

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
DATASET_PATH = "/content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Hosni Youssef/dataset_entrainement (1).jsonl"

OUTPUT_DIR = "./qwen-cnrs-lora"
MAX_SEQ_LENGTH = 1024
BATCH_SIZE = 1
GRAD_ACCUM = 4
EPOCHS = 2
LR = 2e-4

# =========================
# DATASET
# =========================

dataset = load_dataset(
    "json",
    data_files=DATASET_PATH,
    split="train"
)

print("✅ Exemple dataset :")
print(dataset[0])

# =========================
# TOKENIZER
# =========================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# =========================
# MODÈLE (FP16)
# =========================

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

model.config.use_cache = False

# =========================
# LoRA
# =========================

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# =========================
# FORMATAGE PROMPT (INSTRUCTION TUNING)
# =========================

def format_prompt(example):
    return f"""<s>[INST]
Tu es un assistant officiel du CNRS.
Réponds UNIQUEMENT à partir du contexte fourni.
Si l'information n'est pas présente, indique-le explicitement.

QUESTION :
{example["instruction"]}

CONTEXTE :
{example["input"]}
[/INST]

{example["output"]}
</s>
"""

# =========================
# ARGUMENTS D'ENTRAÎNEMENT
# =========================

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none"
)

# =========================
# TRAINER (TRL COMPATIBLE)
# =========================

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    formatting_func=format_prompt
)

# =========================
# ENTRAÎNEMENT
# =========================

trainer.train()

# =========================
# SAUVEGARDE
# =========================

trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("✅ Fine-tuning terminé avec succès.")


ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipython-input-3270383894.py", line 7, in <cell line: 0>
    from datasets import load_dataset
  File "<frozen importlib._bootstrap>", line 1360, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1322, in _find_and_load_unlocked
  File "<frozen importlib._bootstrap>", line 1262, in _find_spec
  File "<frozen importlib._bootstrap_external>", line 1532, in find_spec
  File "<frozen importlib._bootstrap_external>", line 1504, in _get_spec
  File "<frozen importlib._bootstrap_external>", line 1483, in _path_importer_cache
OSError: [Errno 107] Transport endpoint is not connected

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 2099, 

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# =========================
# CONFIG
# =========================

BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
LORA_DIR = "/content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Richel Azebaze - Analyse/qwen-cnrs-lora"   # dossier issu du fine-tuning

MAX_NEW_TOKENS = 200

# =========================
# CHARGEMENT MODELE
# =========================

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True
)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

model = PeftModel.from_pretrained(base_model, LORA_DIR)
model.eval()

print("✅ Modèle fine-tuné chargé avec succès")

# =========================
# FONCTION DE TEST
# =========================

def ask_cnrs(question, context):
    prompt = f"""
Tu es un assistant officiel du CNRS.
Tu dois répondre UNIQUEMENT à partir du CONTEXTE ci-dessous.

RÈGLE :
- Si la réponse n'est pas explicitement dans le contexte, réponds :
"Je ne dispose pas de cette information dans les documents de référence."

CONTEXTE :
{context}

QUESTION :
{question}

RÉPONSE :
""".strip()

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=0.0,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# =========================
# EXEMPLE DE TEST
# =========================

if __name__ == "__main__":
    question = "Quelles sont les principales missions de l'ingénieur biologiste ?"

    context = """
L'ingénieur biologiste devra concevoir et conduire des protocoles d'acquisitions
innovants en IRM in vivo sur une plateforme préclinique et clinique.
Il réalisera des acquisitions IRM chez des modèles animaux et chez l'humain.
"""

    response = ask_cnrs(question, context)

    print("\n❓ QUESTION :\n", question)
    print("\n📘 RÉPONSE :\n", response)


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

✅ Modèle fine-tuné chargé avec succès

❓ QUESTION :
 Quelles sont les principales missions de l'ingénieur biologiste ?

📘 RÉPONSE :
 Tu es un assistant officiel du CNRS.
Tu dois répondre UNIQUEMENT à partir du CONTEXTE ci-dessous.

RÈGLE :
- Si la réponse n'est pas explicitement dans le contexte, réponds :
"Je ne dispose pas de cette information dans les documents de référence."

CONTEXTE :

L'ingénieur biologiste devra concevoir et conduire des protocoles d'acquisitions
innovants en IRM in vivo sur une plateforme préclinique et clinique.
Il réalisera des acquisitions IRM chez des modèles animaux et chez l'humain.


QUESTION :
Quelles sont les principales missions de l'ingénieur biologiste ?

RÉPONSE : 
Concevoir et conduire des protocoles d'acquisition innovants en IRM in vivo sur une plateforme préclinique et clinique, ainsi que réaliser des acquisitions IRM chez des modèles animaux et chez l'humain. Je ne dispose pas de cette information dans les documents de référence.
Je ne dispos

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
LORA_DIR = "/content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Richel Azebaze - Analyse/qwen-cnrs-lora"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

model = PeftModel.from_pretrained(base_model, LORA_DIR)
model.eval()

print("✅ Modèle fine-tuné chargé")


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

✅ Modèle fine-tuné chargé


In [ ]:
# ============================================================
# 🤖 RAG CNRS – Version finale robuste (post fine-tuning)
# ============================================================

import os
import json
import torch
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# =========================
# 1️⃣ PARAMÈTRES
# =========================
JSON_DIR = "/content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Wahib B/cleaned_json_full"
BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
LORA_DIR = "/content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Richel Azebaze - Analyse/qwen-cnrs-lora"

TOP_K = 6
MAX_NEW_TOKENS = 220
REFUS = "Je ne dispose pas de cette information dans les documents de référence."

# =========================
# 2️⃣ CHARGEMENT DU MODÈLE
# =========================
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

model = PeftModel.from_pretrained(base_model, LORA_DIR)
model.eval()

# =========================
# 3️⃣ CHARGEMENT + CHUNKING INTELLIGENT DES JSON
# =========================
documents = []

def add_chunk(text, source, meta):
    if isinstance(text, str) and len(text.strip()) > 40:
        documents.append({
            "text": " ".join(text.split()),
            "source": source,
            "meta": meta
        })

for file in os.listdir(JSON_DIR):
    if not file.endswith(".json"):
        continue

    with open(os.path.join(JSON_DIR, file), "r", encoding="utf-8") as f:
        data = json.load(f)

    # --- CAS 1 : fichiers concours (122)
    if "postes" in data:
        base_meta = {
            "bap": data.get("bap"),
            "grade": data.get("grade"),
            "concours": data.get("concours_label"),
            "concours_num": data.get("concours_num"),
            "emploi_type": data.get("emploi_type")
        }

        for poste in data["postes"]:
            poste_meta = {
                **base_meta,
                "poste_num": poste.get("poste_num"),
                "affectation": poste.get("affectation"),
                "groupe_fonction": poste.get("groupe_fonction")
            }

            add_chunk(poste.get("mission"), file, {**poste_meta, "type": "mission"})
            add_chunk(poste.get("activites"), file, {**poste_meta, "type": "activites"})
            add_chunk(poste.get("competences"), file, {**poste_meta, "type": "competences"})
            add_chunk(poste.get("contexte"), file, {**poste_meta, "type": "contexte"})

    # --- CAS 2 : fichiers transverses (4)
    else:
        for k, v in data.items():
            if isinstance(v, str):
                add_chunk(v, file, {"type": k})

print(f"✅ {len(documents)} chunks indexés (granularité fine)")

# =========================
# 4️⃣ EMBEDDINGS + FAISS
# =========================
embedder = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

embeddings = embedder.encode(
    [d["text"] for d in documents],
    convert_to_numpy=True,
    show_progress_bar=True
).astype("float32")

index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

# =========================
# 5️⃣ RETRIEVAL
# =========================
def retrieve_context(question, k=TOP_K):
    q_emb = embedder.encode(
        [question],
        convert_to_numpy=True
    ).astype("float32")

    _, idx = index.search(q_emb, k)
    return [documents[i] for i in idx[0]]

# =========================
# 6️⃣ GÉNÉRATION CONTRÔLÉE
# =========================
def answer(question):
    contexts = retrieve_context(question)

    context_text = "\n\n".join(
        f"[SOURCE: {c['source']} | {c['meta'].get('type')}]\n{c['text']}"
        for c in contexts
    )

    prompt = f"""
SYSTEM:
Tu es un assistant officiel du CNRS.
Tu peux reformuler et synthétiser.
Tu dois répondre UNIQUEMENT à partir du CONTEXTE.
Si aucune information claire n'est présente, répond exactement :
"{REFUS}"

CONTEXTE :
{context_text}

QUESTION :
{question}

RÉPONSE :
""".strip()

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=0.25,
            repetition_penalty=1.05,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id
        )

    decoded = tokenizer.decode(output[0], skip_special_tokens=True)
    return decoded.split("RÉPONSE:")[-1].strip(), contexts

# =========================
# 7️⃣ INTERFACE CONSOLE
# =========================
print("\n" + "=" * 70)
print("🤖 Agent conversationnel CNRS – Concours Ingénieurs")
print("📌 Basé uniquement sur les documents officiels fournis")
print("✋ Tapez 'stop' pour quitter")
print("=" * 70 + "\n")

while True:
    q = input("❓ Votre question : ").strip()
    if q.lower() == "stop":
        print("👋 Fin de session.")
        break

    response, sources = answer(q)

    print("\n📘 RÉPONSE :\n")
    print(response)

    print("\n📂 SOURCES :")
    for s in sources:
        print(f"- {s['source']} ({s['meta'].get('type')})")

    print("\n" + "-" * 70)

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipython-input-337484329.py", line 9, in <cell line: 0>
    import faiss
  File "<frozen importlib._bootstrap>", line 1360, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1322, in _find_and_load_unlocked
  File "<frozen importlib._bootstrap>", line 1262, in _find_spec
  File "<frozen importlib._bootstrap_external>", line 1532, in find_spec
  File "<frozen importlib._bootstrap_external>", line 1504, in _get_spec
  File "<frozen importlib._bootstrap_external>", line 1483, in _path_importer_cache
OSError: [Errno 107] Transport endpoint is not connected

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 2099, in showtraceback
    s